# README

- **Author**: `李芸榛（JenniferjjLee）`
- **Created At**: `2026-11-08`
- **Last Modified At**: `2026-11-08`（加入 `trip_group`）

---

## What does this file do?

- **加入臺大人文館**：依組員決定，在 `data/processed/stations_in_field.csv`（2026-11-07 版，107 站，全部有 StationID）之外，另外把 `臺大人文館` 納入研究範圍，共 108 個站名，從 24 個原始月檔重新篩選。
  - 臺大人文館 2025-02-04 才出現第一筆交易，不在 2024 年站位快照中，所以 `stations_in_field` 還沒有它的 StationID。這裡先用暫代 ID **`-2`**（表示「在範圍內、ID 待補」）。
  - `stations_in_field` 補上臺大人文館後，把它從 `PENDING_STATIONS` 移除並重跑，就會改用正確 ID。站名清單和 `PENDING_STATIONS` 重疊時，程式會停下來提醒。
- 觀測單位是一筆來源交易紀錄。借車站或還車站任一端與 108 個站名之一（去掉 `YouBike2.0_` 前綴）**精確**相同即保留（OR）；兩端都相同也只輸出一次。不去重、不修正站名、不補值，年度依來源檔月份決定，規則與 v3–v5 相同。
- 讀檔沿用 `YouBike_traffic_raw.py`（stage2）已驗證的 `chunks()` 與 `serialize_selected()`；亂碼 bytes 以 `<UNRESOLVED_UTF8_HEX:...>` 表示。
- **站名**：所有非空站名都加前綴 `YouBike2.0_`；空白站名與亂碼 token 保持原樣。
- **StationID**（`rent_station_id`、`return_station_id`）：
  - 107 站：`stations_in_field` 的 9 碼 StationID（4 個同名移位站合併成一對一對照）。
  - 臺大人文館：`-2`（暫代）。
  - 其他站、空白站名、亂碼 token：`-1`（範圍外，不是真實站點代碼）。
- **與 v5 的關係**：v6 不讀 v5，可以獨立重跑。與 v5 的逐列比對在本 notebook 之外另做一次（見結果摘要）。
- **分組欄 `trip_group`**（2026-11-08 加入，放在最後一欄；依組員定義，四組互斥、先判斷 d）：
  - `a` 外借台大還（流入）：借車站 ID = `-1`，還車站 ID ≠ `-1`。
  - `b` 台大借外還（流出）：借車站 ID ≠ `-1`，還車站 ID = `-1`。
  - `c` 台大借台大還（區內）：兩端 ID 都 ≠ `-1`，且不屬於 d。
  - `d` 無效還車：借還同站，且 `rent` 時長 ≤ 180 秒（3 分鐘）。
  - 「台大」= 研究範圍 108 站，臺大人文館（`-2`）算範圍內。
- 範例檔取 2025 v6 依檔案順序的前 100 列（2026-11-08 由 2024 改為 2025）。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/YYYYMM_YouBike2.0票證刷卡資料.csv`（202401–202512，共 24 個檔）
     - Input key: `raw_YYYYMM`
     - Description: 臺北市 YouBike 2.0 票證刷卡原始資料，合計 133,966,680 筆。
     - Documentation: `data/raw/YouBike_traffic_codebook.md`
     - SHA-256: 每個月檔各自寫在下方 `EXPECTED_RAW_SHA256`，每月筆數寫在 `EXPECTED_RAW_ROWS`（取自第一階段稽核 `stage1_res/`）。
  2. `data/processed/stations_in_field.csv`
     - Input key: `stations_in_field`
     - Description: 研究範圍內站點清單（2026-11-07 版），111 列 × 9 欄、107 站；使用 `StationNameZh_tw` 與 `StationID`。
     - Documentation: pending；產生程式 `data/code/filter_stations_in_field.R`
     - SHA-256: `83919dc097be4e7f82594deff97d782d2fdaf1de52538b08b9a23d764d6aabee`

---

## What does this file output?

- `data/processed/youbike_traffic/YouBike_traffic_2024_v6.csv`
  - Output key: `traffic_2024`
  - Description: 2024（來源月份 202401–202412）借車或還車任一端在 108 個站名中的交易；站名加前綴，附 `rent_station_id`、`return_station_id`。
  - Role: `analysis`
  - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
  - Expected schema and row-count rules: 10 欄 `rent_time, rent_station, rent_station_id, return_time, return_station, return_station_id, rent, bike_type, infodate, trip_group`；每列至少一端在清單內；107 站 ID 與清單一致、臺大人文館為 `-2`、其餘為 `-1`；`trip_group` 符合分組規則；107 站都要出現；UTF-8-SIG。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: `data/processed/youbike_traffic/YouBike_traffic_2024_v6.csv` (same exact file)
- `data/processed/youbike_traffic/YouBike_traffic_2025_v6.csv`
  - Output key: `traffic_2025`
  - Description: 2025（來源月份 202501–202512），規則同上。
  - Role: `analysis`
  - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
  - Expected schema and row-count rules: 同 2024。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: `data/processed/youbike_traffic/YouBike_traffic_2025_v6.csv` (same exact file)
- `data/processed/youbike_traffic/YouBike_traffic_processed_example_v6.csv`
  - Output key: `traffic_example`
  - Description: 2025 v6 依檔案順序的前 100 列（非抽樣、非排序），10 欄。
  - Role: 範例（檢視格式用，不作分析樣本）
  - Documentation: 本 Notebook
  - Expected schema and row-count rules: 100 列 × 10 欄，逐格等於 2025 v6 前 100 列。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: 不建議作分析輸入

All paths are relative to the project root, not the operating-system root.
兩個完整年度檔超過 GitHub 單檔 100 MB 上限，已列入 `data/processed/youbike_traffic/.gitignore`，需透過 repo 以外的管道分享；範例檔可以 commit。


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

In [2]:
import sys
from collections import Counter

BASE_DIR = "data/raw/2024-25YouBike2.0票證刷卡資料"
MONTHS = [f"{year}{month:02}" for year in (2024, 2025) for month in range(1, 13)]
RAW_INPUTS = {month: f"{BASE_DIR}/raw_data/{month}_YouBike2.0票證刷卡資料.csv" for month in MONTHS}
# Fingerprints and row counts of the 24 raw files, as verified by the stage-1 audit
# (stage1_res/input_sha256_before.json and stage1_repair_summary.csv).
EXPECTED_RAW_SHA256 = {
    "202401": "c31144a247f56d4364eb1412d346e6555f3956c83c8193ab200222ee7eb0d71e",
    "202402": "9e8f65f61b5343e5eeb06e9fb9c3603ab2d26a4293c275286aa495b87e95c0fd",
    "202403": "f799f5184ffbeee69cac2f688ce53912ebdba8d904ce8848e17d1b7498471a1f",
    "202404": "70962342ff2a085bf250877d80ddcfa283d9e344552e82a8b46edbaf27e71410",
    "202405": "7e91e909b31efc7c0c4948d904964df1a92b4e8a2b1c3cb26525e30d0a18129c",
    "202406": "dd23954ec298f98203488d278f72e40baf2e6a57539b34e863dc70dfa4471c0d",
    "202407": "12a16422f71a24fd2566fcafcd2e2aef9e32f5f06bf4a637adca55bdcbee7d23",
    "202408": "53130e05fe3988999a68e4dd6ea01784a16e6deb8dec4bcafcc51c506cc748f4",
    "202409": "079fa0a3138a9eadcc32724950f0359b58ac96117868d96f9a424e4e54547706",
    "202410": "00b80c52412b1f023f0f0e04201f2a546c270bde1d6c8846e29287826e3346fc",
    "202411": "fd6ef55345a769bf06fef2f7ff1213028e333468fb8cdf4c32bfb35c7a76b5b1",
    "202412": "0d7de25e428a49036dbbb9dcdf94841dc8ad21623de6a6280e7cb47c77bd4617",
    "202501": "712fba59b84530d6798f8b3457d1fe0f96458956377e2977750f2caef1202a20",
    "202502": "da1618f7922e96b847662895c84be8ad7caf5adaeba89def898c20eca2564114",
    "202503": "e8e355e74dc48cd1e977105bb3505b41812f6d9d52e98aeab0c5549eb67d163a",
    "202504": "f283e3783cb2ca83479c763639e7e89d98d84523f156e395cc5394ce14f25770",
    "202505": "5454436cebe2554148a9af2e054a97f5758692e7e45788a66be693303cde77ba",
    "202506": "43b401d4fa3579335ccd0a467b12ead36d6702ca64e993880b0d9f12b66d0aad",
    "202507": "669781855d96574e1db4f86166990ff77576bc7ba6bc00c211c37e70ac34b217",
    "202508": "23795936c673f7bb7f4f930af6485260707ba7a4520b79fbcaf319cac891ac34",
    "202509": "22791cb3ab8e1c84ce8026a1afb84ee3e9e37ace84f26926dec412334989b38a",
    "202510": "09989ebb841075c6ddbe172bd1b4f02908d84055cf08b56524762307edc21c62",
    "202511": "225636746356f42a58bb85bb9914d711a4d0e3f94acadca46eacff57e692bf85",
    "202512": "904d1279d9491f9f70ce0b5e0dcb800a1cfa3cf367dd5ed1238a6d88765f0eab",
}
EXPECTED_RAW_ROWS = {
    "202401": 2_746_557,
    "202402": 2_590_755,
    "202403": 3_472_416,
    "202404": 3_798_550,
    "202405": 4_338_453,
    "202406": 3_773_353,
    "202407": 5_528_656,
    "202408": 6_180_179,
    "202409": 5_799_896,
    "202410": 5_257_627,
    "202411": 5_883_102,
    "202412": 6_511_640,
    "202501": 5_552_541,
    "202502": 5_252_701,
    "202503": 6_208_631,
    "202504": 6_278_725,
    "202505": 6_608_720,
    "202506": 6_607_567,
    "202507": 6_533_487,
    "202508": 7_011_211,
    "202509": 7_500_365,
    "202510": 6_501_821,
    "202511": 6_829_087,
    "202512": 7_200_640,
}
INPUTS = {
    **{f"raw_{month}": path for month, path in RAW_INPUTS.items()},
    # 2026-11-07 version: 86 polygon stations + 21 manually added stations = 107 stations, all with StationID.
    "stations_in_field": "data/processed/stations_in_field.csv",
}
EXPECTED_INPUT_SHA256 = {
    **{f"raw_{month}": EXPECTED_RAW_SHA256[month] for month in MONTHS},
    "stations_in_field": "83919dc097be4e7f82594deff97d782d2fdaf1de52538b08b9a23d764d6aabee",
}
OUT_DIR = "data/processed/youbike_traffic"
OUTPUTS = {
    "traffic_2024": f"{OUT_DIR}/YouBike_traffic_2024_v6.csv",
    "traffic_2025": f"{OUT_DIR}/YouBike_traffic_2025_v6.csv",
    "traffic_example": f"{OUT_DIR}/YouBike_traffic_processed_example_v6.csv",
}
# All outputs go to data/processed/youbike_traffic/. The two full files exceed GitHub's
# 100 MB limit and are listed in that folder's .gitignore; the small example can be committed.
OUTPUT_AREAS = {"traffic_2024": "processed", "traffic_2025": "processed", "traffic_example": "processed"}
YEAR_OUTPUT = {2024: "traffic_2024", 2025: "traffic_2025"}

# Text reads: empty strings stay "" (not NaN) so blank bike_type / station names are preserved.
TEXT_READ = {"dtype": str, "keep_default_na": False, "na_filter": False, "encoding": "utf-8-sig"}
CHUNK_ROWS = 500_000
TRAFFIC_COLUMNS = ["rent_time", "rent_station", "return_time", "return_station", "rent", "bike_type", "infodate"]
OUTPUT_COLUMNS = ["rent_time", "rent_station", "rent_station_id", "return_time", "return_station",
                  "return_station_id", "rent", "bike_type", "infodate", "trip_group"]

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1]
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, OUTPUT_AREAS[key], must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
print(f"{len(input_fingerprints)} inputs match their expected SHA-256 "
      f"(24 raw months, stations_in_field).")

# Reuse the verified stage-1/stage-2 readers: binary line tracking, surrogateescape for invalid
# UTF-8, quoted fields, per-row column-count checks, and 6-column months padded with bike_type.
traffic_code_dir = project_file(f"{BASE_DIR}/YouBike_traffic_raw.py", "raw").parent
sys.path.insert(0, str(traffic_code_dir))
import YouBike_traffic_raw as stage2  # noqa: E402  (also imports station_dataprocessing as stage1)
assert stage2.COLUMNS == TRAFFIC_COLUMNS

stations = pd.read_csv(input_paths["stations_in_field"], **TEXT_READ)
print({"input": "stations_in_field", "rows": len(stations), "columns": list(stations.columns)})

25 inputs match their expected SHA-256 (24 raw months, stations_in_field).
{'input': 'stations_in_field', 'rows': 111, 'columns': ['StationUID', 'StationID', 'StationNameZh_tw', 'StationNameEn', 'PositionLat', 'PositionLon', 'first_seen', 'last_seen', 'days_observed']}


## 站名 → StationID 對照（107 站 + 臺大人文館暫代 ID -2）

In [3]:
# Station-name -> StationID lookup: the 107 stations of stations_in_field, plus 臺大人文館 with a
# placeholder ID (-2) until stations_in_field gets its real StationID.
PREFIX = "YouBike2.0_"
OUTSIDE_ID = "-1"   # not in the study field
PENDING_ID = "-2"   # in the study field, real StationID not yet available
TOKEN_PREFIX = "<UNRESOLVED_UTF8_HEX:"

assert len(stations) == 111 and stations["StationUID"].nunique() == 107
assert (stations != "").all().all(), "stations_in_field has blank cells."
assert stations["StationNameZh_tw"].str.startswith(PREFIX).all()
assert stations["StationID"].str.fullmatch(r"\d{9}").all()

name_to_id = stations[["StationNameZh_tw", "StationID"]].drop_duplicates()
assert name_to_id["StationNameZh_tw"].is_unique and name_to_id["StationID"].is_unique
assert len(name_to_id) == 107
STATION_ID = dict(zip(name_to_id["StationNameZh_tw"], name_to_id["StationID"]))

# Stations added by the team before they appear in stations_in_field (first trip 2025-02-04; no 2024 snapshot).
# When stations_in_field gains the station, drop it here so its real ID is used (the assert below enforces this).
PENDING_STATIONS = ["臺大人文館"]
for name in PENDING_STATIONS:
    assert PREFIX + name not in STATION_ID, f"{name} is now in stations_in_field; remove it from PENDING_STATIONS."
    STATION_ID[PREFIX + name] = PENDING_ID
assert len(STATION_ID) == 108

TARGETS = {name.removeprefix(PREFIX) for name in STATION_ID}
for name in TARGETS:
    assert name == name.strip() and name and not stage2.stage1.suspicious(name), name
print({"stations with real ID": 107, "stations with placeholder ID -2": PENDING_STATIONS, "filter names": len(TARGETS)})


def add_prefix(names):
    """Prefix every non-empty station name; keep blanks and unresolved-byte tokens unchanged."""
    keep = (names == "") | names.str.startswith(TOKEN_PREFIX) | names.str.startswith(PREFIX)
    return names.where(keep, PREFIX + names)


# trip_group (team definition, 2026-11-08); mutually exclusive, d is checked first:
#   a = 外借台大還（流入）: rent outside (-1), return inside
#   b = 台大借外還（流出）: rent inside, return outside (-1)
#   c = 台大借台大還（區內）: both inside, not d
#   d = 無效還車: same station and ride time (rent) <= 180 seconds
# "Inside" = any ID other than -1, so 臺大人文館 (-2) counts as inside.
TRIP_GROUPS = {"a": "外借台大還（流入）", "b": "台大借外還（流出）", "c": "台大借台大還（區內，不含 d）",
               "d": "無效還車（同站借還且時長 ≤ 3 分鐘）"}
INVALID_MAX_SECONDS = 180


def duration_seconds(rent):
    """rent is 'H:MM:SS' (hours may exceed two digits)."""
    return (rent.str[:-6].astype(np.int64) * 3600 + rent.str[-5:-3].astype(np.int64) * 60
            + rent.str[-2:].astype(np.int64))


def trip_group(frame):
    rent_in = frame["rent_station_id"] != OUTSIDE_ID
    return_in = frame["return_station_id"] != OUTSIDE_ID
    short_same = (frame["rent_station"] == frame["return_station"]) & (duration_seconds(frame["rent"]) <= INVALID_MAX_SECONDS)
    group = np.select([short_same, ~rent_in & return_in, rent_in & ~return_in, rent_in & return_in],
                      ["d", "a", "b", "c"], default="?")
    assert (group != "?").all(), "A row has neither end in the field."
    return group


def transform(chunk):
    out = chunk.copy()
    for side in ("rent", "return"):
        out[f"{side}_station"] = add_prefix(chunk[f"{side}_station"])
        # -1 = outside the field (or blank / unreadable name); -2 = in the field, ID pending.
        out[f"{side}_station_id"] = out[f"{side}_station"].map(STATION_ID).fillna(OUTSIDE_ID)
    out["trip_group"] = trip_group(out)
    return out[OUTPUT_COLUMNS]


demo = pd.DataFrame({c: ["x"] * 5 for c in TRAFFIC_COLUMNS})
demo["rent_station"] = ["臺大人文館", "", "臺大禮賢樓東南側", "臺大禮賢樓東南側", "臺大禮賢樓東南側"]
demo["return_station"] = ["羅斯福路三段283巷", "臺大禮賢樓東南側", "臺大水源舍區A棟", "臺大禮賢樓東南側", "臺大禮賢樓東南側"]
demo["rent"] = ["00:10:00", "00:10:00", "00:05:00", "00:03:00", "00:03:01"]
demo_out = transform(demo)
print(demo_out[["rent_station", "rent_station_id", "return_station", "return_station_id", "rent", "trip_group"]].to_string(index=False))
assert demo_out["rent_station_id"].tolist() == ["-2", "-1", "500119044", "500119044", "500119044"]
assert demo_out["return_station_id"].tolist() == ["-1", "500119044", "500119005", "500119044", "500119044"]
assert demo_out["trip_group"].tolist() == ["b", "a", "c", "d", "c"]

{'stations with real ID': 107, 'stations with placeholder ID -2': ['臺大人文館'], 'filter names': 108}
       rent_station rent_station_id        return_station return_station_id     rent trip_group
   YouBike2.0_臺大人文館              -2 YouBike2.0_羅斯福路三段283巷                -1 00:10:00          b
                                 -1   YouBike2.0_臺大禮賢樓東南側         500119044 00:10:00          a
YouBike2.0_臺大禮賢樓東南側       500119044   YouBike2.0_臺大水源舍區A棟         500119005 00:05:00          c
YouBike2.0_臺大禮賢樓東南側       500119044   YouBike2.0_臺大禮賢樓東南側         500119044 00:03:00          d
YouBike2.0_臺大禮賢樓東南側       500119044   YouBike2.0_臺大禮賢樓東南側         500119044 00:03:01          c


## 從 24 個原始月檔重新篩選並寫出 v6

逐月讀取（沿用 stage2 讀取函式），保留借車或還車任一端在 108 個站名中的交易，同時加前綴與 StationID。每月讀到的 bytes 與筆數都要和第一階段稽核一致。

In [4]:
# Re-filter every raw month: keep a trip when either end exactly matches one of the 108 names (OR).
# Year follows the source-file month, as in stage 2. No dedup, renaming, or imputation.
import time

monthly = []
annual_rows = Counter()
with output_paths["traffic_2024"].open("x", encoding="utf-8-sig", newline="") as out_2024, \
     output_paths["traffic_2025"].open("x", encoding="utf-8-sig", newline="") as out_2025:
    streams = {2024: out_2024, 2025: out_2025}
    for stream in streams.values():
        stream.write(",".join(OUTPUT_COLUMNS) + "\n")
    for month in MONTHS:
        started = time.time()
        year = int(month[:4])
        state = Counter()  # stage2.chunks fills original_rows, source_sha256, csv_read_complete
        stats = Counter()
        for frame in stage2.chunks(input_paths[f"raw_{month}"], month, state):
            rent_hit = frame["rent_station"].isin(TARGETS)
            return_hit = frame["return_station"].isin(TARGETS)
            keep = rent_hit | return_hit
            stats["rent_matched"] += int(rent_hit.sum())
            stats["return_matched"] += int(return_hit.sum())
            stats["both_matched"] += int((rent_hit & return_hit).sum())
            stats["kept"] += int(keep.sum())
            selected = stage2.serialize_selected(frame.loc[keep, TRAFFIC_COLUMNS].copy())
            selected = selected.fillna("").astype(str)  # bike_type is NA for 202401-202410
            transform(selected).to_csv(streams[year], index=False, header=False, lineterminator="\n")
        assert state["csv_read_complete"], f"{month}: read incomplete."
        assert state["source_sha256"] == EXPECTED_RAW_SHA256[month], f"{month}: bytes read differ from audit."
        assert state["original_rows"] == EXPECTED_RAW_ROWS[month], f"{month}: row count differs from audit."
        assert stats["kept"] == stats["rent_matched"] + stats["return_matched"] - stats["both_matched"]
        annual_rows[year] += stats["kept"]
        monthly.append({"month": month, "original_rows": state["original_rows"], **stats,
                        "seconds": round(time.time() - started)})

monthly = pd.DataFrame(monthly)
monthly["retention_rate"] = (monthly["kept"] / monthly["original_rows"]).round(4)
pd.set_option("display.width", 200)
print(monthly.to_string(index=False))
annual = monthly.assign(year=monthly["month"].str[:4]).groupby("year")[["original_rows", "kept"]].sum()
print(annual)
assert monthly["original_rows"].sum() == 133_966_680

 month  original_rows  rent_matched  return_matched  both_matched    kept  seconds  retention_rate
202401        2746557        230041          221666        141544  310163        8          0.1129
202402        2590755        231812          223556        145900  309468        7          0.1195
202403        3472416        406481          396179        276466  526194       10          0.1515
202404        3798550        421422          408975        284556  545841       11          0.1437
202405        4338453        493922          482393        339134  637181       13          0.1469
202406        3773353        364429          354952        243317  476064       11          0.1262
202407        5528656        501823          489074        341577  649320       16          0.1174
202408        6180179        569122          554886        390424  733584       18          0.1187
202409        5799896        743223          730539        552725  921037       18          0.1588
202410    

## 範例檔

In [5]:
# Example file: first 100 rows of the saved 2025 v6 file, in file order.
example_frame = pd.read_csv(output_paths["traffic_2025"], nrows=100, **TEXT_READ)
assert example_frame.shape == (100, 10) and list(example_frame.columns) == OUTPUT_COLUMNS
with output_paths["traffic_example"].open("x", encoding="utf-8-sig", newline="") as stream:
    example_frame.to_csv(stream, index=False, lineterminator="\n")
pd.set_option("display.max_columns", None)
display(example_frame.head(10))

,rent_time,rent_station,rent_station_id,return_time,return_station,return_station_id,rent,bike_type,infodate,trip_group
0,2025-01-13 15:00:00,YouBike2.0_捷運古亭站(8號出口),-1,2025-01-13 21:00:00,YouBike2.0_新生南路三段52號前,500101008,05:34:25,電輔車,2025-01-13,a
1,2025-01-13 01:00:00,YouBike2.0_辛亥路一段30號前,500101012,2025-01-13 03:00:00,YouBike2.0_捷運公館站(4號出口),500106004,01:50:15,一般車,2025-01-13,c
2,2025-01-13 11:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 12:00:00,YouBike2.0_立人國小,-1,00:15:08,一般車,2025-01-13,b
3,2025-01-13 12:00:00,YouBike2.0_臺大男一舍前,500101032,2025-01-13 12:00:00,YouBike2.0_捷運公館站(2號出口),500101022,00:06:22,一般車,2025-01-13,c
4,2025-01-13 17:00:00,YouBike2.0_和平泰順街口,-1,2025-01-13 17:00:00,YouBike2.0_羅斯福路三段245號前,500101016,00:04:40,電輔車,2025-01-13,a
5,2025-01-13 15:00:00,YouBike2.0_臺大卓越研究大樓,500119006,2025-01-13 16:00:00,YouBike2.0_臺大立體機車停車場,500119051,00:35:58,一般車,2025-01-13,c
6,2025-01-13 16:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 17:00:00,YouBike2.0_捷運萬隆站(2號出口),-1,00:52:45,一般車,2025-01-13,b
7,2025-01-13 15:00:00,YouBike2.0_捷運公館站(2號出口),500101022,2025-01-13 16:00:00,YouBike2.0_臺大鹿鳴堂東側,500119061,00:32:42,一般車,2025-01-13,c
8,2025-01-13 15:00:00,YouBike2.0_臺大小福樓東側,500119050,2025-01-13 16:00:00,YouBike2.0_臺大小福樓東側,500119050,00:38:56,一般車,2025-01-13,c
9,2025-01-13 13:00:00,YouBike2.0_臺大工綜館南側,500119052,2025-01-13 13:00:00,YouBike2.0_臺大二號館,500119065,00:03:08,一般車,2025-01-13,c


## 讀回驗證

In [6]:
# Reopen every saved output and check it.
listed = set(STATION_ID)
pending_prefixed = {PREFIX + name for name in PENDING_STATIONS}
verification = {}
group_counts = {}
d_months = Counter()
for year, out_key in YEAR_OUTPUT.items():
    rows, token_cells, blank_cells, pending_rows = 0, 0, 0, 0
    id_seen = set()
    groups = Counter()
    for saved in pd.read_csv(output_paths[out_key], chunksize=CHUNK_ROWS, **TEXT_READ):
        assert list(saved.columns) == OUTPUT_COLUMNS, f"Schema changed while saving {out_key}."
        rent_in, return_in = saved["rent_station"].isin(listed), saved["return_station"].isin(listed)
        assert (rent_in | return_in).all(), f"{out_key}: a row has neither end in the station list."
        for side, in_list in (("rent", rent_in), ("return", return_in)):
            names, ids = saved[f"{side}_station"], saved[f"{side}_station_id"]
            is_token = names.str.startswith(TOKEN_PREFIX)
            assert (names[(names != "") & ~is_token].str.startswith(PREFIX)).all()
            assert (ids[in_list] == names[in_list].map(STATION_ID)).all()
            assert (ids[~in_list] == OUTSIDE_ID).all()
            assert ((ids == PENDING_ID) == names.isin(pending_prefixed)).all(), "-2 must mark exactly 臺大人文館."
            id_seen.update(ids[in_list])
            token_cells += int(is_token.sum())
            blank_cells += int((names == "").sum())
        assert (saved["trip_group"].to_numpy() == trip_group(saved)).all(), f"{out_key}: trip_group differs from its rule."
        groups.update(saved["trip_group"].value_counts().to_dict())
        d_months.update(saved.loc[saved["trip_group"] == "d", "infodate"].str[:7].value_counts().to_dict())
        pending_rows += int((saved["rent_station"].isin(pending_prefixed) | saved["return_station"].isin(pending_prefixed)).sum())
        rows += len(saved)
    assert rows == annual_rows[year], f"{out_key}: row count changed while saving."
    real_seen = id_seen - {PENDING_ID}
    assert len(real_seen) == 107, f"{out_key}: only {len(real_seen)} of 107 stations appear."
    assert sum(groups.values()) == rows
    group_counts[year] = groups
    verification[out_key] = {"data_path": OUTPUTS[out_key], "rows": rows, "columns": len(OUTPUT_COLUMNS),
                             "stations with real ID appearing": len(real_seen),
                             "rows touching 臺大人文館 (ID -2)": pending_rows,
                             "unresolved-token cells": token_cells, "blank-name cells": blank_cells,
                             "sha256": sha256_file(output_paths[out_key]), "status": "read-back checks passed"}
    print(verification[out_key])

example_saved = pd.read_csv(output_paths["traffic_example"], **TEXT_READ)
pd.testing.assert_frame_equal(example_saved, pd.read_csv(output_paths["traffic_2025"], nrows=100, **TEXT_READ))
verification["traffic_example"] = {"data_path": OUTPUTS["traffic_example"], "rows": len(example_saved),
                                   "columns": example_saved.shape[1],
                                   "sha256": sha256_file(output_paths["traffic_example"]),
                                   "status": "read-back checks passed (= first 100 rows of 2025 v6)"}
print(verification["traffic_example"])

group_table = pd.DataFrame({y: {g: group_counts[y].get(g, 0) for g in TRIP_GROUPS} for y in group_counts})
group_table["合計"] = group_table.sum(axis=1)
group_table["占全部"] = (group_table["合計"] / group_table["合計"].sum()).map(lambda x: f"{x:.2%}")
group_table.insert(0, "說明", pd.Series(TRIP_GROUPS))
for col in (2024, 2025, "合計"):
    group_table[col] = group_table[col].map(lambda x: f"{x:,}")
print("trip_group 各組列數")
display(group_table)
assert min(d_months) >= "2025-08", "d appears before 2025-08"
print("d 組各月列數（2025-08 以前資料來源沒有 5 分鐘內的同站還車）：", dict(sorted(d_months.items())))
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."
print("All inputs unchanged after processing.")

{'data_path': 'data/processed/youbike_traffic_processed/YouBike_traffic_2024_v6.csv', 'rows': 7659827, 'columns': 10, 'stations with real ID appearing': 107, 'rows touching 臺大人文館 (ID -2)': 0, 'unresolved-token cells': 781, 'blank-name cells': 1, 'sha256': '22f3366e0a6a8b56fcd135b8b85b3a00f5a5204bafdfa73f47484b4edffb20a7', 'status': 'read-back checks passed'}
{'data_path': 'data/processed/youbike_traffic_processed/YouBike_traffic_2025_v6.csv', 'rows': 10236556, 'columns': 10, 'stations with real ID appearing': 107, 'rows touching 臺大人文館 (ID -2)': 99221, 'unresolved-token cells': 104, 'blank-name cells': 3, 'sha256': '66cbd581b26568f2f8af03d4fafec5b4bc74fe9acf2ffe6feece6aeeb93495b9', 'status': 'read-back checks passed'}
{'data_path': 'data/processed/youbike_traffic_processed/YouBike_traffic_processed_example_v6.csv', 'rows': 100, 'columns': 10, 'sha256': '682a0bf28c68b545b9614845444d907435168d6519ecb6ba4d630680a559081a', 'status': 'read-back checks passed (= first 100 rows of 2025 v6)'}
t

,說明,2024,2025,合計,占全部
a,外借台大還（流入）,"1,681,251","2,207,095","3,888,346",21.73%
b,台大借外還（流出）,"1,824,539","2,399,539","4,224,078",23.60%
c,台大借台大還（區內，不含 d）,"4,154,037","5,627,218","9,781,255",54.65%
d,無效還車（同站借還且時長 ≤ 3 分鐘）,0,"2,704","2,704",0.02%


d 組各月列數（2025-08 以前資料來源沒有 5 分鐘內的同站還車）： {'2025-08': 126, '2025-09': 713, '2025-10': 556, '2025-11': 715, '2025-12': 594}
All inputs unchanged after processing.


## 結果摘要

以下依 2026-11-08 在全新的 in-process IPython shell 由上而下完整執行的實際輸出整理（本機沒有 Jupyter）。本次執行加入 `trip_group` 欄，取代同日稍早未含分組欄的 v6。

- **輸入**：25 個輸入檔的 SHA-256 都符合預期；24 個月逐月讀到的 bytes 和筆數都與第一階段稽核一致，合計 133,966,680 筆。處理後所有輸入檔都沒有變動。
- **輸出與驗證**：三個檔案都已重新讀回，所有 assertion 通過。
  - `YouBike_traffic_2024_v6.csv`：7,659,827 列 × 10 欄；臺大人文館 0 筆（2024 年尚未出現）。
  - `YouBike_traffic_2025_v6.csv`：10,236,556 列 × 10 欄；99,221 列的一端是臺大人文館（ID `-2`）。
  - `YouBike_traffic_processed_example_v6.csv`：100 列 × 10 欄，與 2025 v6 前 100 列逐格相同（範例原取 2024 年，已依組員要求改為 2025 年；兩個年度檔重跑前後 SHA-256 相同）。
  - 107 站在兩個年度都有交易；`-2` 恰好只出現在臺大人文館；`trip_group` 每列都符合分組規則。
- **`trip_group` 各組列數**（兩年合計 17,896,383 列）：
  - a 外借台大還（流入）：3,888,346 列（21.73%）
  - b 台大借外還（流出）：4,224,078 列（23.60%）
  - c 台大借台大還（區內）：9,781,255 列（54.65%）
  - d 無效還車：2,704 列（0.02%），全部在 2025-08 以後
- **d 組只在 2025-08 以後出現**：2025-07 以前，資料來源就沒有 5 分鐘內的同站還車紀錄（同站借還最短幾乎都是 300–301 秒）；2025-08 起才出現 1–6 秒的紀錄（各月：2025-08 有 126 列，之後每月 556–715 列）。所以 d 組不能做跨期比較；2025-08 以後的 c 組也多了 3–5 分鐘的同站借還。
- **與先前版本比對**（2026-11-08 在本 notebook 之外另以程式比對）：
  - 本版去掉 `trip_group` 後，2024、2025 兩檔與加欄前的 v6 逐格、逐列順序相同（當時的 2024 範例檔亦同）。
  - 各組的列與先前另存的 a／b／c／d 分組檔逐列相同（分組檔已依組員決定刪除）。
  - 加欄前的 v6 也已與 v5 比對：2024 逐位元組相同；2025 去掉只碰到臺大人文館的 21,111 列並把 `-2` 換回 `-1` 後，與 v5 的 10,215,445 列相同。
- **範例檔表格的顯示**：本次直接以 `display(...)` 顯示。
- **仍保留的限制**：
  - 臺大人文館暫用 `-2`，等 `stations_in_field` 補上正確 StationID 後需重跑。
  - `羅斯福路三段283巷`（2025-10-10 起才有交易）仍不在範圍內，出現在另一端時 ID 為 `-1`。
  - 亂碼 token 保持原樣：2024 年 781 格、2025 年 104 格；空白站名：2024 年 1 格、2025 年 3 格。

> **資料夾改名**（2026-11-08）：輸出資料夾由 `data/processed/youbike_traffic_processed/` 改名為 `data/processed/youbike_traffic/`，程式中的路徑已同步更新。上方儲存的執行輸出是改名前的執行結果，仍顯示舊路徑；檔案只是搬移，內容與 SHA-256 都沒有改變。
